In [39]:

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold,GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, average_precision_score
from cdc_ml.config import POLLS_PROCESSED,PREFERENCE_PROCESSED


In [40]:
df= pd.read_parquet(POLLS_PROCESSED)

In [41]:
df.head()

,id,username,cycle_start,cycle_end,polling_at,has_booking,cycle_start_month,cycle_start_day,cycle_start_dow,cycle_start_hour,polling_month,polling_day,polling_dow,polling_hour,hours_into_cycle
0,0,ajithak,2025-08-19 09:15:00+08:00,2025-08-21 11:00:00+08:00,2025-08-19 09:00:00+08:00,False,8,19,1,9,8,19,1,9,-0.25
1,0,ajithak,2025-08-19 09:15:00+08:00,2025-08-21 11:00:00+08:00,2025-08-19 10:00:00+08:00,False,8,19,1,9,8,19,1,10,0.75
2,0,ajithak,2025-08-19 09:15:00+08:00,2025-08-21 11:00:00+08:00,2025-08-19 11:00:00+08:00,False,8,19,1,9,8,19,1,11,1.75
3,0,ajithak,2025-08-19 09:15:00+08:00,2025-08-21 11:00:00+08:00,2025-08-19 12:00:00+08:00,False,8,19,1,9,8,19,1,12,2.75
4,0,ajithak,2025-08-19 09:15:00+08:00,2025-08-21 11:00:00+08:00,2025-08-19 13:00:00+08:00,False,8,19,1,9,8,19,1,13,3.75


In [42]:
df["haha"]=df.groupby("id").transform("size")

In [43]:
df.head()

,id,username,cycle_start,cycle_end,polling_at,has_booking,cycle_start_month,cycle_start_day,cycle_start_dow,cycle_start_hour,polling_month,polling_day,polling_dow,polling_hour,hours_into_cycle,haha
0,0,ajithak,2025-08-19 09:15:00+08:00,2025-08-21 11:00:00+08:00,2025-08-19 09:00:00+08:00,False,8,19,1,9,8,19,1,9,-0.25,50
1,0,ajithak,2025-08-19 09:15:00+08:00,2025-08-21 11:00:00+08:00,2025-08-19 10:00:00+08:00,False,8,19,1,9,8,19,1,10,0.75,50
2,0,ajithak,2025-08-19 09:15:00+08:00,2025-08-21 11:00:00+08:00,2025-08-19 11:00:00+08:00,False,8,19,1,9,8,19,1,11,1.75,50
3,0,ajithak,2025-08-19 09:15:00+08:00,2025-08-21 11:00:00+08:00,2025-08-19 12:00:00+08:00,False,8,19,1,9,8,19,1,12,2.75,50
4,0,ajithak,2025-08-19 09:15:00+08:00,2025-08-21 11:00:00+08:00,2025-08-19 13:00:00+08:00,False,8,19,1,9,8,19,1,13,3.75,50


In [44]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32492 entries, 0 to 32491
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype                         
---  ------             --------------  -----                         
 0   id                 32492 non-null  int64                         
 1   username           32492 non-null  str                           
 2   cycle_start        32492 non-null  datetime64[us, Asia/Singapore]
 3   cycle_end          32492 non-null  datetime64[us, Asia/Singapore]
 4   polling_at         32492 non-null  datetime64[us, Asia/Singapore]
 5   has_booking        32492 non-null  bool                          
 6   cycle_start_month  32492 non-null  int32                         
 7   cycle_start_day    32492 non-null  int32                         
 8   cycle_start_dow    32492 non-null  int32                         
 9   cycle_start_hour   32492 non-null  int32                         
 10  polling_month      32492 non-null  int32     

In [45]:
df_pref = pd.read_parquet(PREFERENCE_PROCESSED)

In [46]:

timeslots = ["t_0830", "t_1020", "t_1245", "t_1435", "t_1625", "t_1850", "t_2040"]
pref_long = (
        df_pref.melt(
            id_vars=["id", "username", "day_of_week", "date"],
            value_vars=timeslots,
            var_name="timeslot",
            value_name="selected",
        )
        .query("selected == 1")
        .drop_duplicates(subset=["id", "day_of_week", "timeslot"])
    )


In [47]:
df_pref

,id,username,day_of_week,pref_start,pref_end,date,t_0830,t_1020,t_1245,t_1435,t_1625,t_1850,t_2040,mon,tues,wed,thurs,fri,sat,sun
0,0,ajithak,1,2025-08-19 00:00:00+08:00,2025-08-31 00:00:00+08:00,2025-08-19 00:00:00+08:00,0,0,0,0,0,1,1,"[5, 6]","[5, 6]","[5, 6]","[5, 6]","[5, 6]","[5, 6]","[0, 1, 2, 3, 4, 5, 6]"
1,0,ajithak,2,2025-08-19 00:00:00+08:00,2025-08-31 00:00:00+08:00,2025-08-20 00:00:00+08:00,0,0,0,0,0,1,1,"[5, 6]",[],"[5, 6]",[],"[5, 6]",[],"[0, 1, 2, 3, 4, 5, 6]"
2,0,ajithak,3,2025-08-19 00:00:00+08:00,2025-08-31 00:00:00+08:00,2025-08-21 00:00:00+08:00,0,0,0,0,0,1,1,"[5, 6]","[5, 6]","[5, 6]","[5, 6]","[5, 6]","[0, 1, 2, 3, 4, 5, 6]","[0, 1, 2, 3, 4, 5, 6]"
3,0,ajithak,4,2025-08-19 00:00:00+08:00,2025-08-31 00:00:00+08:00,2025-08-22 00:00:00+08:00,0,0,0,0,0,1,1,"[5, 6]","[5, 6]","[5, 6]","[5, 6]","[0, 1, 2, 3, 4, 5, 6]","[0, 1, 2, 3, 4, 5, 6]","[0, 1, 2, 3, 4, 5, 6]"
4,0,ajithak,5,2025-08-19 00:00:00+08:00,2025-08-31 00:00:00+08:00,2025-08-23 00:00:00+08:00,0,0,0,0,0,1,1,[6],[6],[6],[6],[6],"[2, 3, 4, 5, 6]","[2, 3, 4, 5, 6]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2522,121,addity,2,2025-10-16 00:00:00+08:00,2025-12-14 00:00:00+08:00,2025-12-10 00:00:00+08:00,0,0,0,0,0,1,1,None,None,None,None,None,None,None
2523,121,addity,3,2025-10-16 00:00:00+08:00,2025-12-14 00:00:00+08:00,2025-12-11 00:00:00+08:00,0,0,0,0,0,1,1,None,None,None,None,None,None,None
2524,121,addity,4,2025-10-16 00:00:00+08:00,2025-12-14 00:00:00+08:00,2025-12-12 00:00:00+08:00,0,0,0,0,0,1,1,None,None,None,None,None,None,None
2525,121,addity,5,2025-10-16 00:00:00+08:00,2025-12-14 00:00:00+08:00,2025-12-13 00:00:00+08:00,1,1,1,1,1,1,1,None,None,None,None,None,None,None


In [48]:
id_dates = df_pref.groupby("id")["date"].agg(list).reset_index()

In [49]:
id_dates

,id,date
0,0,"[2025-08-19 00:00:00+08:00, 2025-08-20 00:00:0..."
1,1,"[2025-08-13 00:00:00+08:00, 2025-08-15 00:00:0..."
2,2,"[2025-10-27 00:00:00+08:00, 2025-10-28 00:00:0..."
3,3,"[2026-01-16 00:00:00+08:00, 2026-01-17 00:00:0..."
4,4,"[2025-08-22 00:00:00+08:00, 2025-08-23 00:00:0..."
...,...,...
117,117,"[2026-04-01 00:00:00+08:00, 2026-04-02 00:00:0..."
118,118,"[2026-04-21 00:00:00+08:00, 2026-04-22 00:00:0..."
119,119,"[2025-10-25 00:00:00+08:00, 2025-10-26 00:00:0..."
120,120,"[2025-10-27 00:00:00+08:00, 2025-10-28 00:00:0..."


In [50]:
df = df.merge(id_dates,on="id",how="left")

In [51]:
df.head()

,id,username,cycle_start,cycle_end,polling_at,has_booking,cycle_start_month,cycle_start_day,cycle_start_dow,cycle_start_hour,polling_month,polling_day,polling_dow,polling_hour,hours_into_cycle,haha,date
0,0,ajithak,2025-08-19 09:15:00+08:00,2025-08-21 11:00:00+08:00,2025-08-19 09:00:00+08:00,False,8,19,1,9,8,19,1,9,-0.25,50,"[2025-08-19 00:00:00+08:00, 2025-08-20 00:00:0..."
1,0,ajithak,2025-08-19 09:15:00+08:00,2025-08-21 11:00:00+08:00,2025-08-19 10:00:00+08:00,False,8,19,1,9,8,19,1,10,0.75,50,"[2025-08-19 00:00:00+08:00, 2025-08-20 00:00:0..."
2,0,ajithak,2025-08-19 09:15:00+08:00,2025-08-21 11:00:00+08:00,2025-08-19 11:00:00+08:00,False,8,19,1,9,8,19,1,11,1.75,50,"[2025-08-19 00:00:00+08:00, 2025-08-20 00:00:0..."
3,0,ajithak,2025-08-19 09:15:00+08:00,2025-08-21 11:00:00+08:00,2025-08-19 12:00:00+08:00,False,8,19,1,9,8,19,1,12,2.75,50,"[2025-08-19 00:00:00+08:00, 2025-08-20 00:00:0..."
4,0,ajithak,2025-08-19 09:15:00+08:00,2025-08-21 11:00:00+08:00,2025-08-19 13:00:00+08:00,False,8,19,1,9,8,19,1,13,3.75,50,"[2025-08-19 00:00:00+08:00, 2025-08-20 00:00:0..."


In [52]:
df["valid_days"] = df.apply(lambda row: sum( x > row["polling_at"] for x in row["date"]),axis=1)

    

In [53]:
df.head()

,id,username,cycle_start,cycle_end,polling_at,has_booking,cycle_start_month,cycle_start_day,cycle_start_dow,cycle_start_hour,polling_month,polling_day,polling_dow,polling_hour,hours_into_cycle,haha,date,valid_days
0,0,ajithak,2025-08-19 09:15:00+08:00,2025-08-21 11:00:00+08:00,2025-08-19 09:00:00+08:00,False,8,19,1,9,8,19,1,9,-0.25,50,"[2025-08-19 00:00:00+08:00, 2025-08-20 00:00:0...",12
1,0,ajithak,2025-08-19 09:15:00+08:00,2025-08-21 11:00:00+08:00,2025-08-19 10:00:00+08:00,False,8,19,1,9,8,19,1,10,0.75,50,"[2025-08-19 00:00:00+08:00, 2025-08-20 00:00:0...",12
2,0,ajithak,2025-08-19 09:15:00+08:00,2025-08-21 11:00:00+08:00,2025-08-19 11:00:00+08:00,False,8,19,1,9,8,19,1,11,1.75,50,"[2025-08-19 00:00:00+08:00, 2025-08-20 00:00:0...",12
3,0,ajithak,2025-08-19 09:15:00+08:00,2025-08-21 11:00:00+08:00,2025-08-19 12:00:00+08:00,False,8,19,1,9,8,19,1,12,2.75,50,"[2025-08-19 00:00:00+08:00, 2025-08-20 00:00:0...",12
4,0,ajithak,2025-08-19 09:15:00+08:00,2025-08-21 11:00:00+08:00,2025-08-19 13:00:00+08:00,False,8,19,1,9,8,19,1,13,3.75,50,"[2025-08-19 00:00:00+08:00, 2025-08-20 00:00:0...",12


In [54]:
df.describe()

,id,cycle_start_month,cycle_start_day,cycle_start_dow,cycle_start_hour,polling_month,polling_day,polling_dow,polling_hour,hours_into_cycle,haha,valid_days
count,32492.000000,32492.000000,32492.000000,32492.000000,32492.000000,32492.000000,32492.000000,32492.000000,32492.000000,32492.000000,32492.000000,32492.000000
mean,60.759356,7.361197,14.999446,2.543395,7.544042,7.023175,15.327219,2.972978,11.564970,241.788315,484.955681,16.451434
std,36.137647,4.191270,8.899747,1.788240,8.323271,4.332249,8.671690,1.977111,6.929176,244.844490,348.066636,15.556783
min,0.000000,1.000000,1.000000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,-0.950000,2.000000,0.000000
25%,28.000000,3.000000,8.000000,1.000000,0.000000,2.000000,8.000000,1.000000,6.000000,72.000000,244.000000,4.000000
50%,61.000000,10.000000,14.000000,2.000000,1.000000,9.000000,15.000000,3.000000,12.000000,171.000000,384.000000,11.000000
75%,94.000000,11.000000,22.000000,4.000000,16.000000,11.000000,23.000000,5.000000,18.000000,321.000000,607.000000,27.000000
max,121.000000,12.000000,31.000000,6.000000,23.000000,12.000000,31.000000,6.000000,23.000000,1523.233333,1525.000000,66.000000


In [ ]:
temp = df.groupby("id")[["polling_day","polling_week"]].apply(lambda x :(x>=1).any())
print(temp)

id
0      True
1      True
2      True
3      True
4      True
       ... 
117    True
118    True
119    True
120    True
121    True
Name: polling_day, Length: 122, dtype: bool


In [63]:
df["hi"].value_counts()

hi
True    122
Name: count, dtype: int64